> **Repository execution note.** Start JupyterLab from the **repository root** and place authorised source data in that same root directory (see [README](../README.md) and [Data requirements](../data/README.md)). These notebooks were originally distributed as standalone packages: any older instructions referring to an included ZIP, a `test/` folder, or bundled data do not apply to this GitHub repository. The notebooks preserve their original input schemas and experiment configurations. Results cached in the original files are intentionally excluded from this public-ready copy.

# 03 | Modern Benchmark of Hotel Labour Productivity (2018–2024)

**Luís Silvano's study | Preparation for *Knowledge-Based Systems***

This notebook continues the pandemic-related analyses. It now runs the modern models actually installed in the environment, with temporal comparisons for 2021, 2022, and 2023, outlier handling, spatial ablation, and an analysis of atypical hotels.

**Scientific transparency:** 2024 was examined in previous experiments; its metrics constitute a retrospective audit, not an untouched test. A configuration must never be selected by inspecting 2024. Rigorous confirmation will require a new period or an independent dataset. The historical availability of the geographic variables must also be verified.


## Student guide

1. Place the Excel file `2018- 2024 Luis Silvano - Doctoral Thesis Dataset.xlsx` in the same folder as the notebook.
2. Install the core packages in the **same Jupyter kernel**: `%pip install numpy pandas matplotlib scikit-learn openpyxl lxml xgboost lightgbm catboost`.
3. Install the modern models: `%pip install "pytabkit[models]" pytorch-tabnet tabicl tabpfn causilo`, then restart the kernel. TabR requires FAISS, preferably installed via conda.
4. In the configuration, select `PERFIL='verificacao'` for a quick test or `PERFIL='completo'` to run all architectures. The default is **`completo`**.
5. Run **Kernel > Restart & Run All**. The notebook reports `EXECUTADO` (executed) or `NAO_EXECUTADO` (not executed), with reasons. **A model that is not installed is never assigned fabricated results.**

TabPFN and other foundation models may require a GPU, weight downloads, and authentication/licence acceptance. Check the licences before publishing or using the models.

**Interpretation:** lower MAE and RMSE indicate lower error; higher R² indicates better predictive fit and can be negative. RMSE penalises extreme errors particularly strongly.

**API references:** [PyTabKit](https://github.com/dholzmueller/pytabkit), [TabNet](https://github.com/dreamquark-ai/tabnet), [TabICLv2](https://github.com/soda-inria/tabicl), [TabPFN-3.5](https://github.com/PriorLabs/TabPFN), [Causilo](https://pypi.org/project/causilo/).


## 1. Central configuration


In [ ]:
from pathlib import Path
import json, sys, importlib.util, importlib.metadata, traceback, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor

SEMENTE=42
FICHEIRO_DADOS=Path('2018- 2024 Luis Silvano - Doctoral Thesis Dataset.xlsx')
FOLHA='New Dataset'
PASTA_RESULTADOS=Path('resultados_kbs_modernos')
PASTA_RESULTADOS.mkdir(exist_ok=True,parents=True)
PERFIL='completo'            # 'verificacao' (quick check) or 'completo' (full)
ANOS_CV=[2021,2022,2023]    # each training set uses strictly earlier years only
ANO_AUDITORIA=2024          # year ALREADY explored in previous versions
EXECUTAR_2024=True
MODELOS_COMPLETO=['dummy','ridge','random_forest','lightgbm','xgboost','catboost',
                 'realmlp','tabm','ft_transformer','mlp_plr','tabnet','tabr',
                 'tabicl_v2','tabpfn_35','causilo']
MODELOS_VERIFICACAO=['dummy','ridge','random_forest','lightgbm','xgboost','catboost']
MODELOS=MODELOS_COMPLETO if PERFIL=='completo' else MODELOS_VERIFICACAO
MODELOS_CENARIOS=['random_forest','lightgbm','xgboost','catboost'] if PERFIL=='completo' else ['random_forest','lightgbm']
CENARIOS=['todos_original','todos_p995','sem2020_original','sem2020_p995']
PERMITIR_FOUNDATION_CPU=False  # set to True only if sufficient resources are available
PERMITIR_CHECKPOINT_DOWNLOAD=True
MAX_TABPFN_CPU=5000           # sample limit recommended for CPU use in the documentation
EPOCAS_TABNET=70 if PERFIL=='completo' else 15
THREADS=2

ID_HOTEL='ht_id'; COL_ANO='f_year'; COL_ALVO='f_labprodct'; COL_CADEIA='ht_hotel_chain_d'
ATRIBUTOS_ESTRUTURAIS=['ht_capacity','ht_log_number_of_accommodation_units','ht_stars_classification']
ATRIBUTOS_ESPACIAIS=['ht_location_beach1','ht_location_castle1','ht_location_church1','ht_Restaurants&CoffesL800m']
ATRIBUTOS=ATRIBUTOS_ESTRUTURAIS+ATRIBUTOS_ESPACIAIS+[COL_ANO]
COLUNAS_AUDITORIA=['f_number_of_employees','f_operating_revenue_turnover2']
ATRIBUTOS_HOTEL=ATRIBUTOS_ESTRUTURAIS+ATRIBUTOS_ESPACIAIS+COLUNAS_AUDITORIA
assert PERFIL in {'completo','verificacao'}
assert max(ANOS_CV)<ANO_AUDITORIA
print('File:',FICHEIRO_DADOS.resolve(),'| Profile:',PERFIL,'| Requested models:',MODELOS)

## 2. Library and GPU inventory


In [ ]:
import torch
TEM_GPU=bool(torch.cuda.is_available())
DEVICE='cuda' if TEM_GPU else 'cpu'
MODULOS={'dummy':'sklearn','ridge':'sklearn','random_forest':'sklearn',
         'lightgbm':'lightgbm','xgboost':'xgboost','catboost':'catboost',
         'realmlp':'pytabkit','tabm':'pytabkit','ft_transformer':'pytabkit',
         'mlp_plr':'pytabkit','tabr':'pytabkit','tabnet':'pytorch_tabnet',
         'tabicl_v2':'tabicl','tabpfn_35':'tabpfn','causilo':'causilo'}
inventario=[]
for nome in MODELOS_COMPLETO:
    pacote=MODULOS[nome]
    instalado=importlib.util.find_spec(pacote) is not None
    falta_faiss=nome=='tabr' and importlib.util.find_spec('faiss') is None
    inventario.append({'modelo':nome,'biblioteca':pacote,'instalado':instalado,
                        'falta_faiss':falta_faiss,'selecionado':nome in MODELOS})
inventario=pd.DataFrame(inventario)
print('Python:',sys.version.split()[0],'| PyTorch:',torch.__version__,'| GPU:',TEM_GPU)
display(inventario)
inventario.to_csv(PASTA_RESULTADOS/'inventario.csv',index=False)

## 3. Read the Excel file and filter independent hotels


In [ ]:
# This cell explicitly verifies WHICH file was read and WHICH worksheet was selected.
# The fast reader loads only necessary columns and does not modify the original Excel file.
import re
import zipfile
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

FOLHA_DADOS = 'New Dataset'
COLUNAS_NECESSARIAS = list(dict.fromkeys(
    [ID_HOTEL, COL_ANO, COL_ALVO, COL_CADEIA] + ATRIBUTOS_HOTEL
))

if not FICHEIRO_DADOS.exists():
    raise FileNotFoundError(
        f'Excel file not found: {FICHEIRO_DADOS.resolve()}\n'
        'Place the Excel file next to this notebook or edit FICHEIRO_DADOS in Section 1.\n'
        f'Current Jupyter directory: {Path.cwd()}'
    )

print('Current Jupyter directory:', Path.cwd())
print('File loaded (absolute path):', FICHEIRO_DADOS.resolve())
print('File size (MB):', round(FICHEIRO_DADOS.stat().st_size / 1024**2, 2))

if FICHEIRO_DADOS.suffix.lower() in ('.xlsx', '.xlsm'):
    pasta_excel = load_workbook(FICHEIRO_DADOS, read_only=True, data_only=True)
    folhas = pasta_excel.sheetnames
    print('Worksheets found:', folhas)
    if FOLHA_DADOS not in folhas:
        pasta_excel.close()
        raise ValueError(
            f'Worksheet {FOLHA_DADOS!r} was not found in this Excel file. '
            f'Available worksheets: {folhas}. Please verify that you selected the correct file.'
        )
    folha = pasta_excel[FOLHA_DADOS]
    cabecalho_excel = [str(v).strip() if v is not None else ''
                       for v in next(folha.iter_rows(min_row=1, max_row=1, values_only=True))]
    pasta_excel.close()
    print('Columns in the original Excel file:', len(cabecalho_excel))

    colunas_em_falta = [c for c in COLUNAS_NECESSARIAS if c not in cabecalho_excel]
    if colunas_em_falta:
        print('EXAMPLES OF AVAILABLE COLUMNS:', cabecalho_excel[:25])
        raise ValueError(
            'The following columns were not found in the OPENED file:\n  - '
            + '\n  - '.join(colunas_em_falta)
            + '\nCheck the absolute path above and the Excel file version. '
              'These columns are present in the original Excel file provided by the research team. '
              'These variables will not be silently replaced with others.'
        )

    def read_selected_excel_columns(caminho, nomes, cabecalho):
        'Reads only the requested columns, avoiding the processing of millions of cells in pandas.\n\n        Falls back to pandas.read_excel when lxml is unavailable.\n        Does not modify formulas or values in the original Excel file.\n        '
        try:
            from lxml import etree as ET
        except ImportError:
            print('lxml is not installed; standard reading may take several minutes.')
            return pd.read_excel(
                caminho, sheet_name=FOLHA_DADOS, usecols=lambda c: c.strip() in nomes,
                engine='openpyxl'
            )

        # This project version has one worksheet. For Excel files with multiple
        # worksheets, the standard reader unambiguously resolves the worksheet name.
        if len(folhas) != 1:
            print('Excel file with multiple worksheets: reading with pandas.')
            return pd.read_excel(
                caminho, sheet_name=FOLHA_DADOS, usecols=lambda c: c.strip() in nomes,
                engine='openpyxl'
            )

        ns = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
        # Map an Excel column letter (e.g., 'A', 'HQ') to the variable name.
        letra_para_nome = {
            get_column_letter(i+1): nome
            for i, nome in enumerate(cabecalho)
            if nome in nomes
        }
        linhas = []
        with zipfile.ZipFile(caminho) as arq:
            texto_compartilhado = []
            if 'xl/sharedStrings.xml' in arq.namelist():
                raiz = ET.parse(arq.open('xl/sharedStrings.xml')).getroot()
                texto_compartilhado = [
                    ''.join(si.itertext()) for si in raiz.findall(ns + 'si')
                ]
            with arq.open('xl/worksheets/sheet1.xml') as fluxo:
                parser = ET.iterparse(fluxo, events=('end',), tag=ns+'row', huge_tree=True)
                for _, linha in parser:
                    if linha.get('r') != '1':
                        registo = {}
                        for celula in linha.iterchildren(tag=ns+'c'):
                            # Cell A12 -> column A; the row number is not needed.
                            letra = celula.get('r', '').rstrip('0123456789')
                            nome = letra_para_nome.get(letra)
                            if nome is None:
                                continue
                            valor = celula.findtext(ns+'v')
                            if valor is None:
                                continue
                            tipo = celula.get('t')
                            if tipo == 's':
                                valor = texto_compartilhado[int(valor)]
                            elif tipo == 'b':
                                valor = (valor == '1')
                            elif tipo not in ('str', 'inlineStr'):
                                try:
                                    valor = float(valor)
                                except (ValueError, TypeError):
                                    pass
                            registo[nome] = valor
                        linhas.append(registo)
                    # Avoid storing all three million Excel cells in memory.
                    linha.clear()
                    while linha.getprevious() is not None:
                        del linha.getparent()[0]
        return pd.DataFrame(linhas, columns=nomes)

    dados_brutos = read_selected_excel_columns(
        FICHEIRO_DADOS, COLUNAS_NECESSARIAS, cabecalho_excel
    )
elif FICHEIRO_DADOS.suffix.lower() == '.csv':
    cabecalho_excel = pd.read_csv(FICHEIRO_DADOS, nrows=0).columns.tolist()
    dados_brutos = pd.read_csv(FICHEIRO_DADOS, usecols=COLUNAS_NECESSARIAS)
else:
    raise ValueError('Unrecognised format. Use .xlsx, .xlsm, or .csv.')

dados_brutos.columns = [str(c).strip() for c in dados_brutos.columns]
print('Worksheet used:', FOLHA_DADOS if FICHEIRO_DADOS.suffix.lower() != '.csv' else 'CSV')
print('Rows read:', len(dados_brutos))
print('Columns loaded for the study:', list(dados_brutos.columns))
print('NOTE: only relevant variables are loaded; the original Excel file contains 277 columns.')
display(dados_brutos.head(3))

In [ ]:
# Data conversion is explicit: do not use future values to calculate thresholds.
necessarias = list(dict.fromkeys([ID_HOTEL, COL_ANO, COL_ALVO, COL_CADEIA] +
                                 ATRIBUTOS + COLUNAS_AUDITORIA))
faltam = [c for c in necessarias if c not in dados_brutos.columns]
if faltam:
    raise ValueError('These variables were not found in the opened Excel file: ' + repr(faltam))
painel = dados_brutos[necessarias].copy()
for c in [COL_ANO, COL_CADEIA, COL_ALVO] + ATRIBUTOS_HOTEL:
    painel[c] = pd.to_numeric(painel[c], errors='coerce')
painel = painel.replace([np.inf, -np.inf], np.nan)
qtd_original=len(painel)
painel = painel.loc[(painel[COL_CADEIA] == 0) & (painel[COL_ANO].between(2018, 2024))].copy()
qtd_indep=len(painel)
sem_alvo=int(painel[COL_ALVO].isna().sum())
painel = painel.dropna(subset=[ID_HOTEL, COL_ANO, COL_ALVO]).copy()
if (painel[COL_ALVO] < 0).any():
    raise ValueError('Negative f_labprodct values found; review before applying log1p.')
painel[COL_ANO] = painel[COL_ANO].astype(int)
painel[ID_HOTEL] = painel[ID_HOTEL].astype('Int64').astype(str)
if painel.duplicated([ID_HOTEL, COL_ANO]).any():
    raise ValueError('Duplicate hotel–year pairs exist. Review panel construction before training.')
painel = painel.sort_values([ID_HOTEL,COL_ANO]).reset_index(drop=True)
resumo_anos = painel.groupby(COL_ANO).agg(
    observacoes=(COL_ALVO,'size'),
    hoteis=(ID_HOTEL,'nunique'),
    media=(COL_ALVO,'mean'), mediana=(COL_ALVO,'median'),
    p95=(COL_ALVO,lambda s:s.quantile(.95)),
    p99=(COL_ALVO,lambda s:s.quantile(.99)),
    maximo=(COL_ALVO,'max'),
).reset_index()
print(f'Raw rows: {qtd_original:,} | independent hotels (2018–2024): {qtd_indep:,} | '
      f'missing productivity values: {sem_alvo:,}')
print(f'Usable rows: {len(painel):,} | distinct hotels: {painel[ID_HOTEL].nunique():,}')
display(resumo_anos.round(2))
assert set(painel[COL_ANO].unique()) == set(range(2018,2025))

## 4. Temporal protocol and data leakage prevention

To evaluate year *t*, we use only years preceding *t*. Imputation parameters and outlier thresholds are learned exclusively from the training data. The treatment of 2020 is an experimental hypothesis; neither the observed validation targets nor the 2024 targets are filtered. Same-year financial variables are retained for auditing only and are not included as predictors.

**Important:** location characteristics may reflect the data collection date rather than a historical year. Before publication, verify whether they were available at the prediction date.


In [ ]:
from sklearn.dummy import DummyRegressor

def split_by_year(ano,base=painel):
    treino=base.loc[base[COL_ANO]<ano].copy()
    val=base.loc[base[COL_ANO]==ano].copy()
    if treino.empty or val.empty: raise ValueError('Fold has no observations: '+str(ano))
    assert treino[COL_ANO].max()<val[COL_ANO].min()
    return treino,val

def prepare_training_data(tr,cenario):
    assert cenario in CENARIOS
    if cenario.startswith('sem2020'):
        tr=tr.loc[tr[COL_ANO]!=2020].copy()
    if tr.empty: raise ValueError('Training set is empty after removing 2020')
    y=tr[COL_ALVO].to_numpy(dtype=float).copy()
    limite=None
    if cenario.endswith('p995'):
        limite=float(np.quantile(y,.995))
        y=np.minimum(y,limite)
    return tr,y,limite

def compute_metrics(y,p):
    y=np.asarray(y,dtype=float).reshape(-1)
    p=np.asarray(p,dtype=float).reshape(-1)
    if len(y)!=len(p) or not np.isfinite(p).all():
        raise ValueError('Predictions contain invalid values or have incorrect dimensions')
    return {'MAE':float(mean_absolute_error(y,p)),
            'RMSE':float(np.sqrt(mean_squared_error(y,p))),
            'R2':float(r2_score(y,p))}

def prepare_feature_matrices(tr,va,colunas):
    if any(tr[c].notna().sum()==0 for c in colunas):
        raise ValueError('Feature has no training values: review imputation')
    imp=SimpleImputer(strategy='median')
    a=imp.fit_transform(tr[colunas].astype(float))
    b=imp.transform(va[colunas].astype(float))
    assert np.isfinite(a).all() and np.isfinite(b).all()
    return a.astype(np.float32),b.astype(np.float32)

for ano in ANOS_CV+[ANO_AUDITORIA]:
    tr,va=split_by_year(ano)
    print(ano,'training:',len(tr),'evaluation:',len(va),'last training year:',tr[COL_ANO].max())

## 5. Models actually available and a common interface

Python only reports a model as executed when its `fit` and `predict` operations have completed without errors. All architectures receive the same feature matrix and exactly the same temporal splits. TabNet receives the target as a column vector, as required by its API. Ridge and TabNet use normalisation learned from training data only.


In [ ]:
FALHAS=[]
def record_failure(etapa,nome,ano,cenario,exc):
    FALHAS.append({'etapa':etapa,'modelo':nome,'ano':ano,'cenario':cenario,
                   'erro':str(exc)[:400],'tipo':type(exc).__name__,
                   'traceback':traceback.format_exc(limit=5)[:1200]})
    print('NOT EXECUTED:',etapa,nome,ano,cenario,'-',str(exc)[:135])

def build_model(nome,n_tr):
    if nome=='dummy': return DummyRegressor(strategy='mean')
    if nome=='ridge': return Ridge(alpha=10.0)
    if nome=='random_forest':
        return RandomForestRegressor(n_estimators=180,min_samples_leaf=3,
                                     n_jobs=THREADS,random_state=SEMENTE)
    if nome=='lightgbm':
        from lightgbm import LGBMRegressor
        return LGBMRegressor(n_estimators=300,learning_rate=.045,max_depth=6,num_leaves=20,
                             min_child_samples=18,verbosity=-1,n_jobs=THREADS,random_state=SEMENTE)
    if nome=='xgboost':
        from xgboost import XGBRegressor
        return XGBRegressor(n_estimators=300,max_depth=5,learning_rate=.045,
                            subsample=.9,colsample_bytree=.9,objective='reg:squarederror',
                            tree_method='hist',n_jobs=THREADS,random_state=SEMENTE)
    if nome=='catboost':
        from catboost import CatBoostRegressor
        return CatBoostRegressor(iterations=400,depth=5,learning_rate=.045,
                                 loss_function='RMSE',verbose=False,thread_count=THREADS,
                                 random_seed=SEMENTE,allow_writing_files=False)
    if nome in {'realmlp','tabm','ft_transformer','mlp_plr','tabr'}:
        if nome=='tabr' and importlib.util.find_spec('faiss') is None:
            raise RuntimeError('TabR exige FAISS; instalar via conda num ambiente compativel.')
        import pytabkit
        classe={'realmlp':'RealMLP_TD_Regressor',
                'tabm':'TabM_D_Regressor',
                'ft_transformer':'FTT_D_Regressor',
                'mlp_plr':'MLP_PLR_D_Regressor',
                'tabr':'TabR_S_D_Regressor'}[nome]
        return getattr(pytabkit,classe)(device=DEVICE,random_state=SEMENTE)
    if nome=='tabnet':
        from pytorch_tabnet.tab_model import TabNetRegressor
        return TabNetRegressor(seed=SEMENTE,verbose=0)
    if nome in {'tabicl_v2','tabpfn_35','causilo'}:
        if not TEM_GPU and not PERMITIR_FOUNDATION_CPU:
            raise RuntimeError('GPU recommended. PERMITIR_FOUNDATION_CPU=False; model skipped without execution.')
        if nome=='tabpfn_35' and not TEM_GPU and n_tr>MAX_TABPFN_CPU:
            raise RuntimeError('TabPFN-3.5: >'+str(MAX_TABPFN_CPU)+' samples on CPU. Use a GPU.')
        if not PERMITIR_CHECKPOINT_DOWNLOAD:
            raise RuntimeError('Weight download/authentication disabled in configuration.')
        if nome=='tabicl_v2':
            from tabicl import TabICLRegressor
            return TabICLRegressor(n_estimators=4,random_state=SEMENTE,device=DEVICE,
                                   allow_auto_download=True)
        if nome=='tabpfn_35':
            from tabpfn import TabPFNRegressor
            return TabPFNRegressor()  # v3.5 is the default in recent versions
        from causilo import CausiloRegressor
        return CausiloRegressor(n_estimators=4,random_state=SEMENTE)
    raise ValueError('Unknown model: '+nome)

def run_model(nome,tr,va,cenario,colunas=ATRIBUTOS):
    tr,y,limite=prepare_training_data(tr,cenario)
    Xtr,Xva=prepare_feature_matrices(tr,va,colunas)
    model=build_model(nome,len(tr))
    if nome in {'ridge','tabnet'}:
        scale=StandardScaler()
        Xtr=scale.fit_transform(Xtr).astype(np.float32)
        Xva=scale.transform(Xva).astype(np.float32)
    t0=time.perf_counter()
    if nome=='tabnet':
        model.fit(Xtr,y.astype(np.float32).reshape(-1,1),
                  max_epochs=EPOCAS_TABNET,batch_size=256,virtual_batch_size=128)
    else:
        model.fit(Xtr,y)
    p=np.asarray(model.predict(Xva),dtype=float).reshape(-1)
    p=np.maximum(p,0)  # same physically motivated bound, prespecified for ALL models
    return p,{'segundos':time.perf_counter()-t0,'n_treino':len(tr),
              'limite_y_treino':limite}

print('Training functions are ready. Temporal scenario selection follows.')

## 6. Which treatment should be used: pandemic or robustness?

We compare four scenarios using classical models (without inspecting 2024): keep all years with the original target; keep all years with p99.5 clipping; remove 2020 with the original target; and remove 2020 with p99.5 clipping. The clipping threshold is computed **only from the training data in each fold**. RMSE for each configuration is divided by the RMSE of a Dummy baseline in the same year, so that a year with a much larger target scale does not dominate the decision.


In [ ]:
RMSE_DUMMY={}
for ano in ANOS_CV:
    tr,va=split_by_year(ano)
    pred=np.full(len(va),float(tr[COL_ALVO].mean()))
    RMSE_DUMMY[ano]=compute_metrics(va[COL_ALVO],pred)['RMSE']

linhas_cenario=[]
for nome in MODELOS_CENARIOS:
    for cenario in CENARIOS:
        for ano in ANOS_CV:
            tr,va=split_by_year(ano)
            try:
                pred,info=run_model(nome,tr,va,cenario)
                m=compute_metrics(va[COL_ALVO],pred)
                linhas_cenario.append({'modelo':nome,'cenario':cenario,'ano':ano,**m,
                                       'rmse_relativo':m['RMSE']/RMSE_DUMMY[ano],**info})
            except Exception as e:
                record_failure('cenarios',nome,ano,cenario,e)

cenarios_df=pd.DataFrame(linhas_cenario)
if cenarios_df.empty: raise RuntimeError('No diagnostic ran. Check the error list.')
conta=cenarios_df.groupby(['modelo','cenario'])['ano'].nunique()
completos=set(conta[conta==len(ANOS_CV)].index.tolist())
comuns=set(MODELOS_CENARIOS)
for c in CENARIOS:
    comuns &= {m for m,x in completos if x==c}
if not comuns:
    raise RuntimeError('No model completed every scenario for every year.')
base=cenarios_df.loc[cenarios_df['modelo'].isin(comuns)].copy()
ranking_cenarios=base.groupby('cenario').agg(
    rmse_relativo_medio=('rmse_relativo','mean'),
    RMSE_medio=('RMSE','mean'),MAE_medio=('MAE','mean'),
    modelos=('modelo','nunique'),folds=('ano','size')
).sort_values('rmse_relativo_medio')
CENARIO_ESCOLHIDO=str(ranking_cenarios.index[0])
print('Scenario selected using ONLY 2021–2023:',CENARIO_ESCOLHIDO)
display(ranking_cenarios.round(3))
cenarios_df.to_csv(PASTA_RESULTADOS/'experimentos_cenarios_por_ano.csv',index=False)
ranking_cenarios.to_csv(PASTA_RESULTADOS/'ranking_cenarios.csv')

## 7. Temporal benchmark with recent models

This is the main results table. Only models with **all three validation years completed** enter the overall ranking. Models that are not installed or fail to run receive an explicit reason in a separate report.


In [ ]:
linhas_cv=[]
pred_cv={}
for nome in MODELOS:
    print('\nRequested model:',nome)
    for ano in ANOS_CV:
        tr,va=split_by_year(ano)
        try:
            pred,info=run_model(nome,tr,va,CENARIO_ESCOLHIDO)
            m=compute_metrics(va[COL_ALVO],pred)
            linhas_cv.append({'modelo':nome,'ano':ano,'cenario':CENARIO_ESCOLHIDO,**m,
                              'rmse_relativo':m['RMSE']/RMSE_DUMMY[ano],**info})
            pred_cv[(nome,ano)]=pred
            print(' EXECUTED',ano,'RMSE=',round(m['RMSE'],2),'R2=',round(m['R2'],3))
        except Exception as e:
            record_failure('validacao_modelos',nome,ano,CENARIO_ESCOLHIDO,e)
cv_df=pd.DataFrame(linhas_cv)
if cv_df.empty: raise RuntimeError('No model completed a fold.')
contagem=cv_df.groupby('modelo')['ano'].nunique()
MODELOS_VALIDOS=sorted(contagem[contagem==len(ANOS_CV)].index.tolist())
ranking_modelos=cv_df.loc[cv_df['modelo'].isin(MODELOS_VALIDOS)].groupby('modelo').agg(
    MAE_medio=('MAE','mean'),RMSE_medio=('RMSE','mean'),R2_medio=('R2','mean'),
    rmse_relativo_medio=('rmse_relativo','mean'),segundos=('segundos','sum'),folds=('ano','nunique')
).sort_values('rmse_relativo_medio')
print('Ranking: MODELS WITH ALL FOLDS COMPLETED')
display(ranking_modelos.round(3))
print('Models without complete validation:',sorted(set(MODELOS)-set(MODELOS_VALIDOS)))
cv_df.to_csv(PASTA_RESULTADOS/'cv_modelos_2021_2023.csv',index=False)
ranking_modelos.to_csv(PASTA_RESULTADOS/'ranking_modelos_cv.csv')

In [ ]:
# Heatmap normalised by each year's Dummy baseline so 2022 does not obscure 2023.
from matplotlib.colors import Normalize
d=cv_df[cv_df['modelo'].isin(MODELOS_VALIDOS)].pivot(index='modelo',columns='ano',values='rmse_relativo')
d=d.reindex(ranking_modelos.index)
fig,ax=plt.subplots(figsize=(7.8,max(4.6,.43*len(d)+1.4)))
mat=d.to_numpy(dtype=float)
im=ax.imshow(mat,aspect='auto',cmap='viridis_r',norm=Normalize(vmin=max(0.,np.nanmin(mat)-.03),vmax=np.nanmax(mat)+.03))
ax.set_xticks(range(len(d.columns)),[str(x) for x in d.columns])
ax.set_yticks(range(len(d.index)),d.index)
ax.set(xlabel='Evaluation year',ylabel='Model',title='RMSE relative to Dummy by year (lower is better)')
for r in range(mat.shape[0]):
    for c in range(mat.shape[1]):
        ax.text(c,r,f'{mat[r,c]:.2f}',ha='center',va='center',color='white',fontsize=9)
fig.colorbar(im,ax=ax,shrink=.8,label='RMSE(model) / RMSE(Dummy)')
fig.tight_layout()
fig.savefig(PASTA_RESULTADOS/'figura_rmse_modelos_anos.png',dpi=170,bbox_inches='tight')
plt.show()

## 8. Retrospective audit for 2024

Models that completed validation are retrained on 2018–2023 and applied to 2024. **These results are not used to select models or data treatments.** Because the team has already examined 2024, these metrics must be described as exploratory rather than as a blind confirmation.


In [ ]:
resultados_2024=[]; preds_2024={}
if EXECUTAR_2024:
    tr24,va24=split_by_year(ANO_AUDITORIA)
    for nome in MODELOS_VALIDOS:
        try:
            pred,info=run_model(nome,tr24,va24,CENARIO_ESCOLHIDO)
            m=compute_metrics(va24[COL_ALVO],pred)
            resultados_2024.append({'modelo':nome,'cenario':CENARIO_ESCOLHIDO,
                                    'ano':ANO_AUDITORIA,**m,**info})
            preds_2024[nome]=pred
            print('EXECUTADO 2024:',nome,'RMSE=',round(m['RMSE'],2),'R2=',round(m['R2'],3))
        except Exception as e:
            record_failure('auditoria_2024',nome,2024,CENARIO_ESCOLHIDO,e)
res_2024=pd.DataFrame(resultados_2024)
if not res_2024.empty:
    display(res_2024.sort_values('RMSE').round(3))
    res_2024.to_csv(PASTA_RESULTADOS/'auditoria_2024_exploratoria.csv',index=False)
    observacoes_2024=va24[[ID_HOTEL,COL_ANO,COL_ALVO]].copy()
    for nome,vetor in preds_2024.items(): observacoes_2024['previsao_'+nome]=vetor
    observacoes_2024.to_csv(PASTA_RESULTADOS/'previsoes_2024_identificadores_uso_local.csv',index=False)

## 9. Spatial feature ablation (predefined model)

We use **Random Forest** to compare predictions based only on hotel structure/year with those based on structure/year/spatial context. Feature selection is the same in every fold. This tests whether distances and restaurant counts offer predictive gains, not whether they cause higher productivity. Confirm the collection date of these features before claiming authentic historical forecasting.


In [ ]:
ablacao=[]
for ano in ANOS_CV + ([ANO_AUDITORIA] if EXECUTAR_2024 else []):
    tr,va=split_by_year(ano)
    for grupo,colunas in [('estrutura_ano',ATRIBUTOS_ESTRUTURAIS+[COL_ANO]),
                           ('estrutura_espaco_ano',ATRIBUTOS)]:
        try:
            p,info=run_model('random_forest',tr,va,CENARIO_ESCOLHIDO,colunas)
            ab=compute_metrics(va[COL_ALVO],p)
            ablacao.append({'ano':ano,'preditores':grupo,**ab,'n_treino':info['n_treino']})
        except Exception as e:
            record_failure('ablacao','random_forest',ano,grupo,e)
ablacao_df=pd.DataFrame(ablacao)
if not ablacao_df.empty:
    display(ablacao_df.pivot(index='ano',columns='preditores',values='RMSE').round(2))
    ablacao_df.to_csv(PASTA_RESULTADOS/'ablacao_espacial.csv',index=False)

## 10. Does the previous year's productivity improve predictions?

Models must be compared on **the same subsample**, because not all hotels have complete histories. We create `produtividade_t_menos_1` by joining on the same `ht_id` and the exact previous year; we never use productivity from the forecast year. We also include the *persistence* baseline: predict the value observed in year *t−1* for year *t*.


In [ ]:
lag=painel[[ID_HOTEL,COL_ANO,COL_ALVO]].copy()
lag[COL_ANO]=lag[COL_ANO]+1
lag=lag.rename(columns={COL_ALVO:'produtividade_t_menos_1'})
painel_lag=painel.merge(lag,on=[ID_HOTEL,COL_ANO],how='left',validate='one_to_one')
assert len(painel_lag)==len(painel)
so_lag=painel_lag.loc[painel_lag['produtividade_t_menos_1'].notna()].copy()
res_lag=[]
for ano in ANOS_CV + ([ANO_AUDITORIA] if EXECUTAR_2024 else []):
    tr,va=split_by_year(ano,base=so_lag)
    m=compute_metrics(va[COL_ALVO],va['produtividade_t_menos_1'])
    res_lag.append({'ano':ano,'metodo':'persistencia_t_menos_1','observacoes':len(va),**m})
    for metodo,cols in [('sem_lag_mesmas_linhas',ATRIBUTOS),
                        ('com_lag_t_menos_1',ATRIBUTOS+['produtividade_t_menos_1'])]:
        try:
            p,info=run_model('random_forest',tr,va,CENARIO_ESCOLHIDO,cols)
            res_lag.append({'ano':ano,'metodo':metodo,'observacoes':len(va),**compute_metrics(va[COL_ALVO],p)})
        except Exception as e:
            record_failure('lag','random_forest',ano,metodo,e)
lag_df=pd.DataFrame(res_lag)
if not lag_df.empty:
    display(lag_df.pivot(index='ano',columns='metodo',values='RMSE').round(2))
    lag_df.to_csv(PASTA_RESULTADOS/'experiencia_lag_ano_anterior.csv',index=False)

## 11. Concentration of errors among very high-productivity hotels

For the 2024 diagnostic, we define productivity bands using the 25th, 75th, and 95th percentiles calculated **from training data up to 2023**. This is an error analysis, not a filter applied to the evaluation set. We also report the share of total squared error attributable to the 1% of observations with the largest errors.


In [ ]:
res_faixas=[]
if EXECUTAR_2024 and preds_2024:
    q25,q75,q95=np.quantile(tr24[COL_ALVO],[.25,.75,.95])
    faixa=pd.cut(va24[COL_ALVO],bins=[-np.inf,q25,q75,q95,np.inf],
                 labels=['baixo','medio','alto','extremo'],include_lowest=True)
    for nome,p in preds_2024.items():
        y=va24[COL_ALVO].to_numpy(dtype=float)
        err2=(y-p)**2
        k=max(1,int(np.ceil(.01*len(y))))
        frac=float(np.sort(err2)[-k:].sum()/err2.sum()) if err2.sum() else 0.
        for f in ['baixo','medio','alto','extremo']:
            idx=np.asarray(faixa.astype(str)==f)
            if idx.sum()<2:continue
            res_faixas.append({'modelo':nome,'grupo':f,'observacoes':int(idx.sum()),
                               'erro_quadratico_top1_percent':100*frac,**compute_metrics(y[idx],p[idx])})
faixas_df=pd.DataFrame(res_faixas)
if not faixas_df.empty:
    display(faixas_df[['modelo','grupo','observacoes','MAE','RMSE','erro_quadratico_top1_percent']].round(2))
    faixas_df.to_csv(PASTA_RESULTADOS/'erros_faixas_produtividade_2024.csv',index=False)

## 12. Execution report and reproducibility

**Do not confuse an installed library with an executed model.** Models that fail to complete all years are excluded from the ranking. This section records full failures, results, versions, and training configurations, allowing Luís to repeat the experiments in the same environment.


In [ ]:
status=[]
for nome in MODELOS:
    folds=int(contagem.get(nome,0))
    estado=('EXECUTADO_CV_E_2024' if nome in preds_2024 else 'EXECUTADO_CV') if folds==len(ANOS_CV) else (
        'INCOMPLETO_NAO_RANQUEAR' if folds else 'NAO_EXECUTADO')
    motivos=' | '.join(dict.fromkeys(str(e['erro']) for e in FALHAS if e['modelo']==nome))[:350]
    status.append({'modelo':nome,'status':estado,'folds_executados':folds,
                   'folds_exigidos':len(ANOS_CV),'executou_2024':nome in preds_2024,
                   'motivos':motivos})
status_df=pd.DataFrame(status)
display(status_df)
status_df.to_csv(PASTA_RESULTADOS/'status_modelos_execucao.csv',index=False)
pd.DataFrame(FALHAS,columns=['etapa','modelo','ano','cenario','erro','tipo','traceback']).to_csv(
    PASTA_RESULTADOS/'falhas_modelos.csv',index=False)
versoes={}
for pacote in sorted(set(MODULOS.values())):
    nome_distribuicao={'pytorch_tabnet':'pytorch-tabnet'}.get(pacote,pacote)
    try:versoes[pacote]=importlib.metadata.version(nome_distribuicao)
    except Exception: versoes[pacote]='nao instalado'
meta={'estudo':'Benchmark moderno Luis Silvano',
      'origem':str(FICHEIRO_DADOS.resolve()),'folha':FOLHA,
      'anos_validacao':ANOS_CV,'ano_auditoria':ANO_AUDITORIA,
      'nota_2024':'Exploratory retrospective results; 2024 was inspected in earlier studies',
      'cenario_escolhido_cv':CENARIO_ESCOLHIDO,'modelos_solicitados':MODELOS,
      'modelos_validacao_completa':MODELOS_VALIDOS,'python':sys.version,
      'plataforma':platform.platform(),'gpu':TEM_GPU,'pacotes':versoes,
      'preditores':ATRIBUTOS,'n_observacoes':len(painel),
      'n_hoteis':painel[ID_HOTEL].nunique()}
(PASTA_RESULTADOS/'manifesto_execucao.json').write_text(
    json.dumps(meta,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
print('OUTPUTS IN:',PASTA_RESULTADOS.resolve())

## 13. What this study does—and does not—establish

We can compare architectures only when they **actually run** on the same folds, and discuss the sensitivity of results to 2020 and extreme values. The spatial analysis is a **predictive ablation study**, not a causal conclusion.

**Before submission to KBS**, confirm the accounting definition of `f_labprodct`, review extreme hotels with the doctoral student, validate the historical collection date of restaurant counts and spatial features, add an evaluation on hotels not observed during training, and obtain a new temporal sample for confirmation. Adjustments made after examining 2024 must be reported as exploratory.
